# **Classification: Random Forest Classifier**

## **Justification of Preprocessing & Regularization Strategy**

### **Scale Invariance (Original Data)**
The Random Forest Classifier is a powerful ensemble method composed of multiple Decision Trees. Since individual trees partition data based on discrete feature thresholds rather than computing spatial distances, they are inherently scale-invariant. The model's splitting logic and final predictions remain identical regardless of whether the data is in its raw form, standardized, or normalized. To preserve the direct clinical interpretability of the features and maximize computational efficiency, we will train the model using the **Original, Unscaled Data**.

### **The Scale Problem: Why the Decision Tree Champion Was Not Enough**
Initially, we attempted to lock in our regularized Decision Tree champion parameters (`max_depth=10`, `min_samples_leaf=5`) as the foundation for the forest. However, diagnostic metrics revealed a severe **Train vs. Test gap** (Training Accuracy spiking to 1.0, while Test dropped to ~0.86). 

*The clinical diagnosis:* In a massive dataset of over 100,000 patients, allowing a leaf to contain only 5 patients means isolating groups that represent just 0.005% of the data. When this micro-segmentation is multiplied by hundreds of trees in a forest, the ensemble effectively memorizes individual patients (capturing noise) rather than extracting generalizable medical patterns.

### **Aggressive Pruning at Scale**
To break this extreme overfitting loop and build a deployment-ready model, our preprocessing and hyperparameter strategy underwent a radical shift towards macro-level regularization:
* **Macro-Leaves (`min_samples_split` & `min_samples_leaf`):** We scaled our pruning thresholds drastically. Forcing nodes to split only when they have over 50-100 samples, and leaves to contain at least 50 patients, ensures every clinical rule represents a statistically robust cohort.
* **Shallow Trees (`max_depth`):** Hard-capping depth (e.g., between 5 and 12) to prevent the generation of deep, highly specific memorization paths.
* **Data Starvation (`max_samples`):** Restricting each tree to only see a fraction of the dataset (e.g., 40% to 80%) per bootstrap. This dramatically increases the ensemble's overall diversity and forces the forest to learn generalized trends.

---

## **Experiment Design**

We designed a highly regularized tournament of 3 optimization levels. We strictly log **both Train and Test metrics** across all runs to visually confirm that the Overfitting Gap is structurally closed:

* **Regularized Baseline**: A manually constrained forest (`max_depth=8`, `min_samples_leaf=50`, `max_samples=0.5`) designed to establish a mathematically safe floor where Training Accuracy is physically prevented from hitting 100%.
* **GridSearchCV**: A systematic 3-fold cross-validated search testing discrete combinations of our new macro-pruning thresholds to find the ideal balance between underfitting and overfitting.
* **Optuna Optimization**: Bayesian optimization deployed with large-scale boundaries for leaf sizes and row subsampling to surgically pinpoint the maximum generalizable Recall.

In [3]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_RandomForest")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) maintaining class proportion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_classification_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to explicitly monitor the Overfitting Gap"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics (Vigilance)
    mlflow.log_metric("recall_train", recall_score(y_tr, y_tr_pred))
    mlflow.log_metric("accuracy_train", accuracy_score(y_tr, y_tr_pred))
    mlflow.log_metric("f1_train", f1_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics (True Performance)
    mlflow.log_metric("recall_test", recall_score(y_te, y_te_pred))
    mlflow.log_metric("accuracy_test", accuracy_score(y_te, y_te_pred))
    mlflow.log_metric("f1_test", f1_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: REGULARIZED BASELINE (Aggressive Pruning for 100k rows)
# ---------------------------------------------------------
with mlflow.start_run(run_name="RF_Regularized_Baseline"):
    rf_base = RandomForestClassifier(
        n_estimators=100,
        criterion='gini',
        max_depth=8,                # Raso para evitar caminhos de memorização
        min_samples_split=100,      # Exige 100 pacientes para tentar uma divisão
        min_samples_leaf=50,        # Cada folha tem de representar uma coorte de no mínimo 50 pacientes
        max_features='sqrt',        
        max_samples=0.5,            # Subamostragem drástica (cada árvore só vê metade dos dados)
        random_state=42,
        n_jobs=-1
    )
    
    start_time = time.time()
    rf_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    mlflow.log_params(rf_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_classification_metrics(rf_base, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV (Macro-Pruning Balance)
# ---------------------------------------------------------
with mlflow.start_run(run_name="RF_GridSearch_Ensemble"):
    # Grelha de pesquisa focada estritamente em zonas de alta regularização
    param_grid = {
        'max_depth': [6, 8, 10],
        'min_samples_leaf': [50, 100],
        'max_samples': [0.5, 0.7] 
    }
    
    grid = GridSearchCV(
        RandomForestClassifier(
            n_estimators=150,
            criterion='gini',
            max_features='sqrt',
            random_state=42, 
            n_jobs=-1
        ),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    best_rf_grid = grid.best_estimator_
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_classification_metrics(best_rf_grid, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA (Large-Scale Regularization Search)
# ---------------------------------------------------------
def objective(trial):
    # Optuna confinado a limites macro para lidar com as 100.000 linhas
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 250),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 50, 200),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 30, 150),
        "max_samples": trial.suggest_float("max_samples", 0.4, 0.8),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"])
    }
    
    model = RandomForestClassifier(
        criterion='gini', 
        random_state=42,
        n_jobs=-1,
        **params
    )
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="RF_Optuna_Regularized"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=15) 
    duration = time.time() - start_time
    
    # Treino do campeão final com o ponto de equilíbrio ideal encontrado
    best_rf_opt = RandomForestClassifier(
        criterion='gini', 
        random_state=42,
        n_jobs=-1,
        **study.best_params
    )
    best_rf_opt.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_classification_metrics(best_rf_opt, X_train, y_train, X_test, y_test, duration)

print("Regularized Random Forest tournament completed.")

[I 2026-05-20 22:40:00,583] A new study created in memory with name: no-name-2e87e053-cac0-46d7-971e-130982f9d572
[I 2026-05-20 22:40:03,216] Trial 0 finished with value: 0.870536245754943 and parameters: {'n_estimators': 171, 'max_depth': 5, 'min_samples_split': 134, 'min_samples_leaf': 148, 'max_samples': 0.5730594770529532, 'max_features': 'log2'}. Best is trial 0 with value: 0.870536245754943.
[I 2026-05-20 22:40:06,673] Trial 1 finished with value: 0.8683486572077421 and parameters: {'n_estimators': 145, 'max_depth': 12, 'min_samples_split': 116, 'min_samples_leaf': 89, 'max_samples': 0.5529173208304164, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.870536245754943.
[I 2026-05-20 22:40:10,633] Trial 2 finished with value: 0.8687236585099068 and parameters: {'n_estimators': 205, 'max_depth': 7, 'min_samples_split': 128, 'min_samples_leaf': 58, 'max_samples': 0.6884887291313861, 'max_features': 'log2'}. Best is trial 0 with value: 0.870536245754943.
[I 2026-05-20 22:40:15,1

Regularized Random Forest tournament completed.


## **Winner Run Selection (Priority Elimination Framework)**

### **Re-evaluation using latest logged runs (manual analysis)**
I rechecked the latest `runs.csv` values and applied the mandatory generalization filter plus the Recall→F1→Fit Time decision flow.

### **Generalization filter (mandatory)**
A run is eligible only if both Recall and F1 Train→Test gaps (Test − Train) are within ±0.5 percentage points (|gap| ≤ 0.005). All three recent runs pass this filter.

### **All Runs: Summary Table (Train/Test metrics shown, latest)**

| Run | Accuracy (Train) | Accuracy (Test) | Recall (Train) | Recall (Test) | F1 (Train) | F1 (Test) | Fit Time (s) |
|---|---:|---:|---:|---:|---:|---:|---:|
| RF_Regularized_Baseline | 0.92014 | 0.91815 | 0.86874 | 0.86617 | 0.92884 | 0.92700 | 0.92 |
| RF_GridSearch_Ensemble | 0.92103 | 0.91875 | 0.86924 | 0.86592 | 0.92961 | 0.92748 | 36.01 |
| **RF_Optuna_Regularized** | **0.90554** | **0.90485** | **0.86843** | **0.86783** | **0.91689** | **0.91628** | **48.98** |


### **Step 1 — Filter by Highest Test Recall (Priority 1 — 70%)**
- Best Recall (Test): **0.86783** (`RF_Optuna_Regularized`).
- Differences to best: all runs are within 0.5pp of the best Recall, so we proceed to Priority 2.

### **Step 2 — Verify F1 (Priority 2 — 30%)**
- RF_Regularized_Baseline: F1 (Test) = **0.92700**
- RF_GridSearch_Ensemble: F1 (Test) = **0.92748**
- RF_Optuna_Regularized: F1 (Test) = **0.91628**

Result: **RF_GridSearch_Ensemble** has the highest F1 among the top candidates and therefore wins.

### **Step 3 — Tiebreaker: Fit Time**
- Not required; `RF_GridSearch_Ensemble` (36.01s) is slower than the baseline (0.92s) but wins on F1.

### **Final Decision**
**Winner: RF_GridSearch_Ensemble**

**Justification:** After enforcing the generalization filter, all runs remained eligible. Recall differences were below the 0.5pp threshold, so we resolved the selection using F1; `RF_GridSearch_Ensemble` has the highest Test F1 and is selected despite longer training time.

### **Winner Hyperparameters (approx., from GridSearch run logs)**

| Parameter | Value |
|---|---|
| **n_estimators** | 150 |
| **criterion** | gini |
| **max_depth** | 10 |
| **min_samples_leaf** | 50 |
| **max_samples** | 0.7 |
| **max_features** | sqrt |
| **random_state** | 42 |
| **n_jobs** | -1 |

### **Overfitting / Underfitting Diagnosis**
- Under the ±0.5pp rule for Recall and F1 gaps, none of the current runs are disqualified for overfitting/underfitting.
- Operational recommendation: if you prefer to penalize long fit times more strongly, consider reducing the allowed Recall tie margin or applying a small preference toward faster models in the tie-breaking step.